# Aula 3 — Otimização e Avaliação de Modelos
## Interpretabilidade com SHAP

Este notebook reconstrói, em ordem lógica e executável célula a célula, o conteúdo dos
slides 18 a 64 da Aula 3. As Aulas 1 e 2 trataram de **melhorar** o modelo (features certas,
hiperparâmetros certos); esta aula trata de **entender** as decisões do modelo já treinado —
por que ele previu o que previu, quais features pesaram mais, e se há algo suspeito no seu
comportamento.

**Dataset usado em toda a aula:** `load_breast_cancer` — o mesmo da Aula 1. Aqui o foco
não é mais selecionar features nem ajustar hiperparâmetros, e sim **abrir a caixa-preta**
de um modelo já treinado (XGBoost) e entender suas decisões feature a feature, instância a
instância.

### Roteiro da aula

1. **Fundamentos do SHAP** — o que são SHAP values e por que eles são matematicamente
   "justos" (propriedade de eficiência).
2. **Explicações locais** — por que o modelo previu isso *para esta instância específica*
   (force plot, waterfall plot, decision plot).
3. **Explainers para diferentes modelos** — `TreeExplainer` (rápido, para árvores) vs.
   `KernelExplainer` (lento, mas funciona com qualquer modelo).
4. **Visões globais** — quais features importam mais *no dataset inteiro* (summary/bar plot).
5. **Dependence plots e interações** — como o valor de uma feature afeta sua contribuição,
   e como duas features interagem entre si.
6. **Pipeline completo e aplicações** — juntando tudo, e usando SHAP para investigar viés.

> **Nota sobre um bug real que encontrei ao testar o material original:** o slide 38 (SHAP
> com Random Forest) fazia `shap_values_rf[1]` esperando pegar "os SHAP values da classe
> positiva". Na versão atual do SHAP (0.52), `shap_values` para classificação com árvores
> vem como um array 3D `(n_amostras, n_features, n_classes)` — `[1]` nesse array pega a
> **2ª amostra**, não a classe 1! A indexação correta é `[:, :, 1]`. Esse notebook já usa a
> forma corrigida em todo lugar (e explica o porquê).

## 0. Setup do ambiente (slide 18)

Dataset, split, e um modelo XGBoost treinado — o "paciente" que vamos interpretar ao longo
da aula. Todas as seções seguintes reaproveitam `X_train`, `X_test`, `y_train`, `y_test` e
`model` definidos aqui.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=100, max_depth=5,
    learning_rate=0.1, random_state=42
)
model.fit(X_train, y_train)
print(f'Acurácia no teste: {model.score(X_test, y_test):.4f}')

---
# Bloco 1 — Fundamentos do SHAP

**SHAP** (SHapley Additive exPlanations) atribui a cada feature uma "contribuição" para a
previsão de uma instância específica, baseada em um conceito da teoria dos jogos (valores
de Shapley): quanto cada "jogador" (feature) contribuiu para o "placar final" (a previsão),
considerando todas as ordens possíveis de combinação entre features.

## 1.1 Criando o explainer (slide 19)

`TreeExplainer` é otimizado para modelos baseados em árvore (XGBoost, Random Forest,
LightGBM) — rápido e **exato** (não é uma aproximação). Calcula um SHAP value para **cada
feature de cada instância** do conjunto de teste.

> O `isinstance(shap_values, list)` abaixo é um resquício de versões antigas do SHAP, onde
> `shap_values` para classificação binária às vezes vinha como lista `[classe_0, classe_1]`.
> Na versão atual, para XGBoost binário, `shap_values` já vem como array 2D direto — o
> `if` nunca dispara aqui, mas não faz mal deixá-lo (é útil se você trocar de modelo/versão).

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

expected_value = explainer.expected_value
if isinstance(expected_value, list):
    expected_value = expected_value[1]

print(f'Shape dos SHAP values: {shap_values.shape}')
print(f'Valor base (expected value): {expected_value:.4f}')
print('Cada linha = 1 instância, cada coluna = 1 feature')

## 1.2 Verificando a eficiência (slide 20)

**Propriedade de Eficiência** dos valores de Shapley: a soma dos SHAP values de uma
instância, somada ao valor base (`expected_value` — a previsão média do modelo sem
conhecer nenhuma feature), deve reconstruir **exatamente** a previsão do modelo para essa
instância. Não sobra nem falta nada — é uma explicação completa, sem resíduo.

**Bug real do material original, corrigido aqui:** o slide comparava `shap_values[i].sum()
+ expected_value` diretamente com `predict_proba` — testei e a diferença **não** batia
(ficava na casa de 1 a 6, não perto de zero). O motivo: para XGBoost, o `TreeExplainer`
calcula os SHAP values no **espaço log-odds (margin)** do modelo por padrão — o mesmo
espaço de `model.predict(..., output_margin=True)` — não no espaço de probabilidade de
`predict_proba`. A propriedade de eficiência vale exatamente nesse espaço "natural" do
modelo; para comparar com `predict_proba`, é preciso aplicar a função `sigmoid` (a inversa
do link logístico) na soma antes de comparar.

In [ ]:
from scipy.special import expit as sigmoid

for i in range(5):
    prediction = model.predict_proba(X_test.iloc[[i]])[0][1]
    raw_sum = shap_values[i].sum() + expected_value       # espaço log-odds (margin)
    prob_sum = sigmoid(raw_sum)                             # convertido para probabilidade
    print(f'Instância {i}:')
    print(f'  Previsão do modelo (probabilidade): {prediction:.6f}')
    print(f'  Valor base + Σ SHAP (log-odds):     {raw_sum:.6f}')
    print(f'  sigmoid(Valor base + Σ SHAP):        {prob_sum:.6f}')
    print(f'  Diferença:                           {abs(prediction - prob_sum):.10f}')
    print()

---
# Bloco 2 — Explicações locais

"Por que o modelo previu isso **para esta instância**?" — o tipo de pergunta que SHAP
responde melhor que qualquer feature importance global.

## 2.1 Force Plot: implementação (slide 29)

Visualiza a "força" de cada feature empurrando a previsão pra cima (vermelho) ou pra baixo
(azul), a partir do valor base, até chegar na previsão final daquela instância.

> Lembrando da seção 1.2: para este modelo XGBoost, `expected_value` e `shap_values` estão
> em **log-odds (margin)**, não em probabilidade — por isso `Soma SHAP` abaixo não é
> diretamente a probabilidade prevista. O eixo do force plot também é nessa escala; a
> direção e a magnitude relativa das barras continuam corretas e interpretáveis, só o
> número absoluto não é "probabilidade direta" sem passar pelo sigmoid.

In [ ]:
instance = X_test.iloc[0]

shap.force_plot(
    expected_value,
    shap_values[0],
    instance,
    matplotlib=True,
    show=False
)
plt.savefig('force_plot.png', bbox_inches='tight', dpi=150)
plt.show()

pred = model.predict_proba(instance.values.reshape(1, -1))[0]
print(f'Previsão: classe {pred.argmax()} ({pred.max():.4f})')
print(f'Valor base: {expected_value:.4f}')
print(f'Soma SHAP: {shap_values[0].sum():.4f}')

## 2.2 Interpretando o Force Plot (slide 30)

Além do gráfico, dá pra listar textualmente as features que mais empurraram a previsão em
cada direção — útil pra logs, relatórios ou quando um gráfico não é prático.

In [ ]:
contribs = pd.DataFrame({
    'feature': X_test.columns,
    'shap_value': shap_values[0],
    'feature_value': instance.values
}).sort_values('shap_value', key=abs, ascending=False)

print('=== Top 10 contribuições ===')
for _, row in contribs.head(10).iterrows():
    direction = 'para cima' if row['shap_value'] > 0 else 'para baixo'
    print(f"{row['feature']:25s} = {row['feature_value']:.4f}"
          f"  ->  SHAP = {row['shap_value']:+.4f} ({direction})")

## 2.3 Waterfall Plot: implementação (slide 33)

Mesma informação do Force Plot, em formato de "cascata" — geralmente mais fácil de ler
quando há muitas features, porque lista uma por linha em vez de empilhar horizontalmente.

In [ ]:
explanation = shap.Explanation(
    values=shap_values[0],
    base_values=expected_value,
    data=instance.values,
    feature_names=X_test.columns.tolist()
)

shap.waterfall_plot(explanation, show=False)
plt.savefig('waterfall_plot.png', bbox_inches='tight', dpi=150)
plt.show()

shap.waterfall_plot(explanation, max_display=10, show=False)
plt.savefig('waterfall_top10.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.4 Comparando instâncias (slide 34)

Olhar o waterfall de uma previsão positiva ao lado de uma negativa ajuda a entender quais
features "empurram" o modelo pra cada lado — e se faz sentido clinicamente/no domínio do
problema.

In [ ]:
preds = model.predict(X_test)

pos_idx = np.where(preds == 1)[0][0]
neg_idx = np.where(preds == 0)[0][0]

for idx, label in [(pos_idx, 'Positivo'), (neg_idx, 'Negativo')]:
    exp = shap.Explanation(
        values=shap_values[idx],
        base_values=expected_value,
        data=X_test.iloc[idx].values,
        feature_names=X_test.columns.tolist()
    )
    shap.waterfall_plot(exp, max_display=10, show=False)
    plt.title(f'Previsão: {label}')
    plt.savefig(f'waterfall_{label.lower()}.png', bbox_inches='tight', dpi=150)
    plt.show()

## 2.5 Decision Plot (slide 35)

Enquanto force/waterfall mostram **uma** instância, o Decision Plot sobrepõe várias linhas
(uma por instância), cada uma acumulando as contribuições das features da esquerda pra
direita até chegar na previsão. Bom pra ver **padrões entre várias instâncias** de uma vez —
por exemplo, comparar só as previsões erradas.

In [ ]:
shap.decision_plot(
    expected_value, shap_values,
    X_test, show=False
)
plt.savefig('decision_plot.png', bbox_inches='tight', dpi=150)
plt.show()

correct = preds == y_test
print(f'Previsões incorretas: {(~correct).sum()} de {len(y_test)}')

shap.decision_plot(
    expected_value, shap_values[~correct],
    X_test[~correct],
    title='Previsões INCORRETAS',
    show=False
)
plt.savefig('decision_incorrect.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.6 Force Plot interativo (slide 36)

Versão em HTML/JavaScript do force plot, empilhando várias instâncias lado a lado — dá pra
passar o mouse e explorar. Ideal para dashboards ou para você explorar livremente no
notebook (renderiza inline automaticamente).

In [ ]:
shap.initjs()

shap.force_plot(
    expected_value,
    shap_values[:50],
    X_test.iloc[:50]
)

In [ ]:
force_html = shap.force_plot(
    expected_value,
    shap_values[:50],
    X_test.iloc[:50]
)
shap.save_html('force_plot_interativo.html', force_html)
print('Salvo em force_plot_interativo.html')

---
# Bloco 3 — Explainers para diferentes modelos

Nem todo modelo é uma árvore. Vamos ver como o SHAP se comporta com Random Forest (que
revela uma pegadinha real de indexação) e com um modelo totalmente diferente (rede neural).

## 3.1 Importância média absoluta (slide 37)

Antes de trocar de modelo, uma ponte entre explicação local e global: a **média do valor
absoluto** dos SHAP values de cada feature, através de todas as instâncias do teste, dá uma
medida de importância global — parecida em espírito com a feature importance da Aula 1, mas
derivada da mesma matemática usada nas explicações individuais.

In [ ]:
importancia = pd.DataFrame({
    'feature': X_test.columns,
    'importance': np.abs(shap_values).mean(0)
}).sort_values('importance', ascending=False)

print('=== Top 15 Features (SHAP) ===')
for _, row in importancia.head(15).iterrows():
    bar = '#' * int(row['importance'] * 50)
    print(f"{row['feature']:25s}  {row['importance']:.4f}  {bar}")

shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.savefig('bar_plot_shap.png', bbox_inches='tight', dpi=150)
plt.show()

## 3.2 SHAP com Random Forest (slide 38)

**Bug real do material original, corrigido aqui:** para RandomForest binário, o SHAP atual
(0.52) devolve um array **3D**: `(n_amostras, n_features, n_classes)`. O slide original fazia
`shap_values_rf[1]`, achando que pegaria a classe positiva — mas isso pega a **2ª amostra**
inteira (shape `(30, 2)`), não a classe 1! A forma correta é `shap_values_rf[:, :, 1]`
(shape `(114, 30)`), fatiando o **último eixo** (classes), mantendo todas as amostras.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
print(f'RF Acurácia: {rf.score(X_test, y_test):.4f}')

explainer_rf = shap.TreeExplainer(rf)
shap_values_rf_raw = explainer_rf.shap_values(X_test)
print(f'Shape bruto (amostras, features, classes): {shap_values_rf_raw.shape}')

# CORRETO: fatiar o último eixo (classe positiva = índice 1)
shap_values_rf = shap_values_rf_raw[:, :, 1]
print(f'Shape após selecionar a classe positiva: {shap_values_rf.shape}')

exp_rf = shap.Explanation(
    values=shap_values_rf[0],
    base_values=explainer_rf.expected_value[1],
    data=X_test.iloc[0].values,
    feature_names=X_test.columns.tolist()
)
shap.waterfall_plot(exp_rf, show=False)
plt.savefig('waterfall_rf.png', bbox_inches='tight', dpi=150)
plt.show()

## 3.3 KernelExplainer (slide 39)

`TreeExplainer` só funciona com modelos de árvore. `KernelExplainer` é **model-agnostic** —
funciona com qualquer modelo que tenha `.predict` ou `.predict_proba` (aqui, uma rede
neural `MLPClassifier`), ao custo de ser bem mais lento: em vez de usar a estrutura interna
da árvore, ele estima os SHAP values fazendo perturbações repetidas nas features e
observando como a previsão muda, usando uma amostra do treino como "background"
(referência do que é um valor "típico" de cada feature).

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
print(f'MLP Acurácia: {mlp.score(X_test, y_test):.4f}')

background = shap.sample(X_train, 100)
explainer_kernel = shap.KernelExplainer(mlp.predict_proba, background)

# Mais lento que TreeExplainer - por isso limitamos a poucas instâncias
shap_values_mlp = explainer_kernel.shap_values(X_test.iloc[:20], silent=True)
print(f'Shape: {shap_values_mlp.shape} (amostras, features, classes)')

**Comparando os dois explainers:** o `TreeExplainer` calculou SHAP values **exatos** para
as 114 instâncias do teste quase instantaneamente. O `KernelExplainer` levou alguns segundos
só para **20** instâncias, usando um modelo aproximado. Essa é a razão de o `TreeExplainer`
ser sempre preferível quando o modelo é baseado em árvore — o `KernelExplainer` é o
"último recurso" para modelos sem suporte dedicado (redes neurais, SVM, etc.).

---
# Bloco 4 — Visões globais

Explicações locais (uma instância por vez) são ótimas para casos individuais, mas para
entender o modelo **como um todo** precisamos agregar os SHAP values de todas as instâncias.

## 4.1 Summary Plot — o que é? (slide 50)

O **beeswarm plot** é a visão global mais completa do SHAP: cada ponto é uma instância,
a posição horizontal é o SHAP value (impacto na previsão), e a cor é o valor da própria
feature (vermelho = alto, azul = baixo). Dá pra ver de uma vez: quais features importam
mais (ordenadas de cima pra baixo), a direção do efeito, e se o efeito é consistente ou
varia entre instâncias.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig('summary_plot.png', bbox_inches='tight', dpi=150)
plt.show()

## 4.2 Bar Plot — versão executiva (slide 52)

Quando o beeswarm é informação demais para uma apresentação, o bar plot simplifica: só a
importância média absoluta por feature, sem direção nem distribuição. Bom para slides
executivos onde "quais são as 15 features que mais importam" já responde a pergunta.

In [ ]:
shap.summary_plot(
    shap_values, X_test,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.savefig('bar_plot.png', bbox_inches='tight', dpi=150)
plt.show()

# Versão manual, com mais controle sobre o visual
importancia_manual = pd.DataFrame({
    'feature': X_test.columns,
    'importance': np.abs(shap_values).mean(0)
}).sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(8, 6))
plt.barh(importancia_manual['feature'], importancia_manual['importance'])
plt.xlabel('Mean |SHAP value|')
plt.title('Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('custom_bar.png', dpi=150)
plt.show()

---
# Bloco 5 — Dependence plots e interações

O beeswarm mostra a distribuição geral, mas não a **forma exata** da relação entre o valor
de uma feature e seu impacto. O Dependence Plot preenche essa lacuna.

## 5.1 Dependence Plot — o que é? (slide 54)

Para uma feature específica, plota o **valor da feature** (eixo X) contra seu **SHAP value**
(eixo Y) em cada instância — revela se a relação é linear, tem um ponto de corte, satura,
ou é mais complexa. O SHAP escolhe automaticamente uma segunda feature (cor dos pontos)
que parece interagir mais com a primeira.

In [ ]:
top_feature = pd.Series(
    np.abs(shap_values).mean(0), index=X_test.columns
).idxmax()
print(f'Feature mais importante: {top_feature}')

shap.dependence_plot(
    top_feature, shap_values, X_test,
    show=False
)
plt.savefig('dependence_plot.png', bbox_inches='tight', dpi=150)
plt.show()

## 5.2 Dependence Plot — múltiplas features (slide 55)

O mesmo gráfico para as top 6 features, lado a lado — uma forma rápida de varrer o
comportamento das features mais importantes sem gerar um gráfico de cada vez.

In [ ]:
top_features = pd.Series(
    np.abs(shap_values).mean(0), index=X_test.columns
).sort_values(ascending=False).head(6).index

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.flatten(), top_features):
    shap.dependence_plot(
        feat, shap_values, X_test,
        ax=ax, show=False
    )
    ax.set_title(feat, fontsize=12)

plt.tight_layout()
plt.savefig('dependence_grid.png', dpi=150)
plt.show()

## 5.3 Detectando interações (slide 56)

Em vez de deixar o SHAP escolher a feature de cor automaticamente, podemos **forçar** uma
interação específica com `interaction_index`. Se a cor (segunda feature) muda claramente o
padrão vertical dos pontos para um mesmo valor da primeira feature, há uma interação real
entre as duas — o efeito de uma depende do valor da outra.

In [ ]:
shap.dependence_plot(
    'worst radius',
    shap_values, X_test,
    interaction_index='worst concave points',
    show=False
)
plt.savefig('interaction_plot.png', bbox_inches='tight', dpi=150)
plt.show()

print("Interpretação: se, para um mesmo valor de 'worst radius', os pontos vermelhos")
print("('worst concave points' alto) ficam sistematicamente mais acima ou abaixo dos")
print("pontos azuis, há interação -- 'worst radius' tem um efeito que depende do valor")
print("de 'worst concave points', não é um efeito isolado e constante.")

---
# Bloco 6 — Pipeline completo e aplicações

Fechamos consolidando o fluxo típico de uma análise SHAP e vendo uma aplicação prática além
de "explicar previsões": usar SHAP para investigar se o modelo trata subgrupos de forma
diferente.

## 6.1 Pipeline completo: todas as visualizações (slide 57)

O roteiro típico de uma análise SHAP em produção: treinar → explainer → visão global
(beeswarm + bar) → dependence plots das features mais importantes → explicações locais de
instâncias representativas → decision plot. Nesta ordem: do geral para o específico.

In [ ]:
# 1. Modelo já treinado (model, do setup)
# 2. Explainer (já calculado: explainer, shap_values, expected_value)

# 3. Visão global
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig('pipeline_summary.png', bbox_inches='tight', dpi=150)
plt.show()

shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.savefig('pipeline_bar.png', bbox_inches='tight', dpi=150)
plt.show()

# 4. Dependence (top 3 features)
for feat in top_features[:3]:
    shap.dependence_plot(feat, shap_values, X_test, show=False)
    plt.savefig(f'pipeline_dependence_{feat.replace(" ", "_")}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# 5. Explicação local (instâncias representativas)
for idx in [0, 10, 50]:
    exp = shap.Explanation(
        values=shap_values[idx],
        base_values=expected_value,
        data=X_test.iloc[idx].values,
        feature_names=X_test.columns.tolist()
    )
    shap.waterfall_plot(exp, show=False)
    plt.savefig(f'pipeline_waterfall_{idx}.png', bbox_inches='tight', dpi=150)
    plt.show()

# 6. Decision plot
shap.decision_plot(expected_value, shap_values, X_test, show=False)
plt.savefig('pipeline_decision.png', bbox_inches='tight', dpi=150)
plt.show()

## 6.2 SHAP: detecção de viés (slide 64)

**Nota sobre o material original:** o comentário do slide fala em "supondo uma coluna
`gender`" — mas o `breast_cancer` não tem nenhuma coluna demográfica (idade, gênero etc.).
O código na verdade usa `worst texture` (dividida na mediana) como um **grupo de exemplo
qualquer**, só para ilustrar a técnica. A ideia central continua válida e é importante: se
o SHAP médio de uma feature difere muito entre dois grupos (demográficos ou não), o modelo
está usando essa feature de forma diferente para cada grupo — o que pode ser um sinal de
comportamento inconsistente ou, em contextos reais com colunas sensíveis (raça, gênero,
idade), de viés.

In [ ]:
group_a = shap_values[X_test['worst texture'] > X_test['worst texture'].median()]
group_b = shap_values[X_test['worst texture'] <= X_test['worst texture'].median()]

print('=== Comparação de grupos (acima vs. abaixo da mediana de "worst texture") ===')
for i, feat in enumerate(X_test.columns[:5]):
    diff = np.abs(group_a[:, i].mean() - group_b[:, i].mean())
    status = 'ATENCAO' if diff > 0.1 else 'ok'
    print(f'[{status}] {feat}: diferença de {diff:.4f} entre grupos')

---
# Resumo — progressão lógica da aula

1. **Fundamentos** (slides 18–20): o que são SHAP values e por que a propriedade de
   eficiência garante que a explicação é matematicamente completa (soma exatamente igual à
   previsão), não uma aproximação qualquer de importância.
2. **Explicações locais** (slides 29–36): force plot, waterfall plot e decision plot
   respondem "por que o modelo previu isso para esta instância" — de formas visuais
   diferentes para o mesmo tipo de informação.
3. **Explainers por tipo de modelo** (slides 37–39): `TreeExplainer` é rápido e exato para
   árvores; `KernelExplainer` é o recurso universal (mas lento) para qualquer modelo — e
   RandomForest revelou uma armadilha real de indexação em versões atuais do SHAP.
4. **Visões globais** (slides 50, 52): beeswarm para análise completa, bar plot para
   comunicação executiva — ambos agregando os mesmos SHAP values locais.
5. **Dependence plots e interações** (slides 54–56): a forma da relação entre valor da
   feature e seu impacto, e como duas features podem interagir (o efeito de uma depende do
   valor da outra).
6. **Pipeline completo e aplicações** (slides 57, 64): do geral para o específico numa
   análise real, e uma aplicação prática de SHAP além de "explicar previsões" — auditar se
   o modelo trata subgrupos de forma consistente.

Essa progressão fecha o ciclo das três primeiras aulas da disciplina: Aula 1 escolheu as
features certas, Aula 2 ajustou os hiperparâmetros certos, e esta aula garante que você
consegue **explicar** as decisões do modelo resultante — essencial tanto para debugar
problemas quanto para justificar decisões automatizadas em contextos regulados ou
sensíveis.

---
# Aplicação a outro dataset — Wine (multi-classe)

Até aqui, todo SHAP foi calculado sobre um problema **binário** (`breast_cancer`). Vamos
aplicar a mesma técnica ao **Wine** (o mesmo dataset multi-classe usado na Aula 1) para ver
como o SHAP se comporta com **3 classes** — o array de SHAP values ganha mais uma dimensão,
o mesmo padrão de fatiamento por classe do bloco 3.2 (Random Forest) se repete aqui.

## Setup — Wine

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = wine.target

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=42, stratify=yw)

model_w = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42)
model_w.fit(Xw_train, yw_train)
print(f'Classes: {list(wine.target_names)}')
print(f'Acurácia no teste: {model_w.score(Xw_test, yw_test):.4f}')

## Explainer multi-classe

Com 3 classes, `shap_values` vem como array `(n_amostras, n_features, n_classes)` — mesma
estrutura 3D que vimos no Random Forest binário (bloco 3.2), só que agora com 3 fatias em
vez de 2. Para explicar a previsão de uma classe específica, fatiamos o último eixo.

In [ ]:
explainer_w = shap.TreeExplainer(model_w)
shap_values_w = explainer_w.shap_values(Xw_test)
print(f'Shape (amostras, features, classes): {shap_values_w.shape}')
print(f'Valores base por classe: {explainer_w.expected_value}')

## Visão global por classe

A importância das features pode ser (e geralmente é) diferente para cada classe — uma
feature pode ser decisiva para distinguir a classe 0 das outras, mas irrelevante para
distinguir 1 de 2. Olhamos o bar plot da classe 0 (`class_names[0]`) como exemplo.

In [ ]:
classe_idx = 0
shap_values_classe0 = shap_values_w[:, :, classe_idx]

print(f'Explicando a classe: {wine.target_names[classe_idx]}')
shap.summary_plot(shap_values_classe0, Xw_test, plot_type='bar', show=False)
plt.title(f'Importância das features - classe {wine.target_names[classe_idx]}')
plt.savefig('wine_bar_classe0.png', bbox_inches='tight', dpi=150)
plt.show()

## Explicação local — uma instância

Waterfall plot da primeira instância do teste, explicando a contribuição de cada feature
para a previsão da classe 0 especificamente (o mesmo padrão de fatiamento usado acima).

In [ ]:
exp_w = shap.Explanation(
    values=shap_values_classe0[0],
    base_values=explainer_w.expected_value[classe_idx],
    data=Xw_test.iloc[0].values,
    feature_names=Xw_test.columns.tolist()
)
shap.waterfall_plot(exp_w, show=False)
plt.savefig('wine_waterfall_classe0.png', bbox_inches='tight', dpi=150)
plt.show()

pred_w = model_w.predict_proba(Xw_test.iloc[[0]])[0]
print(f'Probabilidades previstas: {dict(zip(wine.target_names, np.round(pred_w, 4)))}')
print(f'Classe real: {wine.target_names[yw_test[0]]}')

**O que comparar com o `breast_cancer`:**

- O código de análise SHAP (explainer, waterfall, summary_plot) é **exatamente o mesmo** —
  a única mudança real foi lembrar que, com múltiplas classes, `shap_values` ganha uma
  dimensão extra e você precisa escolher **qual classe** está explicando a cada gráfico.
- Isso é uma continuação direta do padrão que já apareceu no bloco 3.2 (Random Forest
  binário) — o "bug corrigido" ali (fatiar `[:, :, classe]` em vez de `[classe]`) é
  exatamente a mesma lição aplicada aqui, só que com 3 classes em vez de 2.